In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 17. text text — text text

> **Learning Goals**
> - "Attention is All You Need" text text text text text
> - Pre-LN vs Post-LNtext text Training text text
> - text-text, text text, text text textdegreestext Comparisontext

## 17.1 text text

2017text Vaswani et al. — RNN text Attentiontext text text.

text:
1. **text**: RNNtext text text text
2. **text text**: text text text text
3. **text**: Model text text text → LLM text text

## 17.2 text text text

| text | text | text Model |
|---|---|---|
| text-text | text text | text (text text, T5, BART) |
| text text | textDirection Attention | BERT, RoBERTa |
| text text | text Attention | GPT, LLaMA, Qwen |

LLM text **text text**text text. text text text text text Training.

## 17.3 text text text

text text:
1. Multi-Head Self-Attention
2. text text (Residual)
3. text text (LayerNorm/RMSNorm)
4. Feed-Forward Network (FFN)
5. text text
6. text

**Post-LN** (text):
$$\mathbf{x}' = \mathrm{LayerNorm}(\mathbf{x} + \mathrm{Attn}(\mathbf{x}))$$

**Pre-LN** (GPT-2 text text):
$$\mathbf{x}' = \mathbf{x} + \mathrm{Attn}(\mathrm{LayerNorm}(\mathbf{x}))$$

Pre-LNtext text Modeltext Training text text.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

class TransformerBlock(nn.Module):
    """Pre-LN Transformer Block."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, pre_ln=True):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.pre_ln = pre_ln

    def attention(self, x, mask=None):
        B, T, D = x.shape
        qkv = self.W_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask=None):
        if self.pre_ln:
            # Pre-LN
            x = x + self.dropout(self.attention(self.ln1(x), mask))
            x = x + self.dropout(self.ffn(self.ln2(x)))
        else:
            # Post-LN
            x = self.ln1(x + self.dropout(self.attention(x, mask)))
            x = self.ln2(x + self.dropout(self.ffn(x)))
        return x

# Test
d_model, n_heads, d_ff = 64, 8, 256
block = TransformerBlock(d_model, n_heads, d_ff)
x = torch.randn(2, 10, d_model)
out = block(x)
print(f"text: {x.shape}")
print(f"Output: {out.shape}")
print(f"Block Parameter Count: {sum(p.numel() for p in block.parameters()):,}")


## 17.4 Position-wise Feed-Forward Network (FFN)

$$\mathrm{FFN}(\mathbf{x}) = \mathrm{GELU}(\mathbf{x} W_1 + \mathbf{b}_1) W_2 + \mathbf{b}_2$$

- $W_1 \in \mathbb{R}^{d \times d_{ff}}$, $W_2 \in \mathbb{R}^{d_{ff} \times d}$
- text $d_{ff} = 4d$ (text: $d=512, d_{ff}=2048$)
- text text **text** text (position-wise)

**SwiGLU** (LLaMA): GLU text Performance text
$$\mathrm{SwiGLU}(\mathbf{x}) = \mathrm{SiLU}(\mathbf{x} W_1) \odot (\mathbf{x} W_3) W_2$$


In [ ]:
# FFN text Comparison
class StandardFFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff)
        self.w2 = nn.Linear(d_ff, d)

    def forward(self, x):
        return self.w2(F.gelu(self.w1(x)))

class SwiGLUFFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d, bias=False)
        self.w3 = nn.Linear(d, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

d, d_ff = 64, 256
std_ffn = StandardFFN(d, d_ff)
glu_ffn = SwiGLUFFN(d, d_ff)
print(f"Standard FFN params: {sum(p.numel() for p in std_ffn.parameters()):,}")
print(f"SwiGLU FFN params: {sum(p.numel() for p in glu_ffn.parameters()):,}")

x = torch.randn(2, 10, d)
y1 = std_ffn(x)
y2 = glu_ffn(x)
print(f"Standard FFN Output: {y1.shape}")
print(f"SwiGLU FFN Output: {y2.shape}")
print("\n=> SwiGLUtext W3 text text text text, Performance Improvement. LLaMA Standard.")


## 17.5 text textResult text text

**text text**: $\mathbf{x}_{\ell+1} = \mathbf{x}_\ell + f_\ell(\mathbf{x}_\ell)$

- text text text text → text Model Training text
- text text text

**text**: textValue Distribution text → text text Trainingtext, text text text


In [ ]:
# text text text: text text text text
def test_grad_flow(depth, use_residual):
    """Depthtext Gradient text text."""
    d = 64
    layers = [nn.Linear(d, d) for _ in range(depth)]
    x = torch.randn(1, d, requires_grad=True)
    target = torch.randn(1, d)
    loss_fn = nn.MSELoss()

    out = x
    for layer in layers:
        if use_residual:
            out = out + layer(out)
        else:
            out = layer(out)
    loss = loss_fn(out, target)
    loss.backward()
    return x.grad.norm().item()

print(f"{'depth':>6} | {'text X':>15} | {'text O':>15}")
print("-" * 40)
for d in [5, 10, 20, 50]:
    g_no = test_grad_flow(d, False)
    g_yes = test_grad_flow(d, True)
    print(f"{d:>6} | {g_no:>15.6e} | {g_yes:>15.6e}")
print("\n=> text Connectiontext textPlane textdegrees Gradienttext text text.")


## 17.6 Model text text Calculation

text text:
- QKV text: $3 d^2$
- Output text: $d^2$
- FFN (standard): $2 \cdot 4d^2 = 8d^2$
- LayerNorm: $4d$
- **text**: $\approx 12 d^2$ per block

text Model ($L$ text):
- text: $|V| \cdot d$
- text: $L \cdot 12 d^2$
- Outputtext: $|V| \cdot d$ (weight tying text text)

text $P \approx 12 L d^2$ (text text text).


In [ ]:
# text text Calculation
def count_transformer_params(vocab_size, d_model, n_layers, d_ff=None, tie_weights=True):
    if d_ff is None: d_ff = 4 * d_model
    # text
    emb = vocab_size * d_model
    # text
    qkv = 3 * d_model * d_model
    out_proj = d_model * d_model
    ffn = 2 * d_model * d_ff
    ln = 4 * d_model  # text text LN
    block = qkv + out_proj + ffn + ln
    # Outputtext
    output = 0 if tie_weights else vocab_size * d_model
    total = emb + n_layers * block + output
    return total, {'emb': emb, 'block': block, 'n_layers': n_layers, 'output': output}

# text Magnitude Comparison
print(f"{'Model':>12} | {'V':>8} | {'d':>5} | {'L':>3} | {'Params':>12}")
print("-" * 50)
for name, V, d, L in [
    ('Mini', 1000, 64, 4),
    ('Small', 10000, 256, 6),
    ('GPT-1', 40000, 768, 12),
    ('GPT-2', 50257, 768, 12),
    ('GPT-2 medium', 50257, 1024, 24),
    ('GPT-3 6.7B', 50257, 4096, 32),
    ('LLaMA-7B', 32000, 4096, 32),
]:
    n, _ = count_transformer_params(V, d, L)
    print(f"{name:>12} | {V:>8} | {d:>5} | {L:>3} | {n:>12,}")


## 17.7 Key Takeaways

| Contents text | text |
|---|---|
| Multi-Head Attention | text text text Training |
| FFN | text text (position-wise) |
| text text | text Model Training text |
| LayerNorm/RMSNorm | textValue text |
| Positional Encoding | text text text |

| text | textdegrees |
|---|---|
| Post-LN | text text |
| Pre-LN | GPT-2+ text |
| SwiGLU FFN | LLaMA text |

## Exercises

1. Pre-LNtext Post-LN text text text Trainingtext text Comparisontext.
2. Standard FFNtext SwiGLU FFNtext text text text Comparisontext (d_ff text text).
3. text text text text text text text 10, 20, 50text Trainingtext textdegreestext text Comparisontext.
4. GPT-2 small (d=768, L=12, V=50257)text text text Calculationtext.
5. text text Attentiontext FFNtext text text Calculationtext.

> Solutions: `solutions/ch17_solutions.ipynb`
